# ORPO: Monolithic Preference Optimization (2024)

## Why ORPO Matters

**Standard DPO** requires a reference model (frozen copy of your SFT model) loaded alongside the policy during training.
On a T4 (16GB), training a 1B model with DPO means: policy (2GB) + reference (2GB) + optimizer (6GB) = 10GB+ — tight.

**ORPO** (Hong et al., March 2024) eliminates the reference model entirely by combining SFT and preference optimization into a single loss:

```
L_ORPO = L_SFT + λ · L_OR

L_SFT = -log P(y_w | x)                         # standard NLL on chosen response
L_OR  = -log σ( log[P(y_w)/(1-P(y_w))] - log[P(y_l)/(1-P(y_l))] )  # odds ratio
```

The odds ratio directly maximizes the gap between the model assigning high probability to chosen responses vs. rejected ones — without ever needing a separate reference model.

**Memory comparison (1B model, T4 GPU):**
| Method | Models in memory | Approx VRAM |
|--------|-----------------|-------------|
| DPO    | policy + reference | ~10GB |
| ORPO   | policy only | ~5GB |
| KTO    | policy + reference | ~10GB |

## TRL 1.3+ API

In TRL ≥ 1.3, ORPO is implemented via `DPOTrainer` with `loss_type="orpo"` in older versions,
but in TRL 1.3+ use the dedicated `ORPOTrainer` from `trl.trainer` or implement from scratch.
We implement from scratch here to make the math transparent.

In [ ]:
!pip install -U trl transformers datasets peft -q
import trl; print(f"TRL {trl.__version__}")

## ORPO Loss — The Math

The key insight: instead of comparing to a reference model, compare the model's own odds ratio for chosen vs rejected.

Odds ratio = P(y) / (1 - P(y)) — how much more likely is y than not-y?

If the model strongly prefers chosen, the log odds gap will be large → sigmoid pushes toward 0 → small OR loss.

In [ ]:
import torch
import torch.nn.functional as F

def orpo_loss(chosen_logps: torch.Tensor, rejected_logps: torch.Tensor, lam: float = 0.1):
    """
    ORPO loss (Hong et al. 2024).
    
    Args:
        chosen_logps:   mean log-probs of chosen response tokens, shape (B,)
        rejected_logps: mean log-probs of rejected response tokens, shape (B,)  
        lam: weight for the odds ratio term (λ in paper, typically 0.1)
    
    Returns:
        total_loss, sft_component, odds_ratio_component
    """
    # SFT component: maximize log-likelihood of chosen response
    sft_loss = -chosen_logps.mean()
    
    # Odds ratio component
    # Clamp to avoid log(0) or log(negative)
    chosen_lp   = chosen_logps.clamp(max=-1e-7)
    rejected_lp = rejected_logps.clamp(max=-1e-7)
    
    log_odds_chosen   = chosen_lp   - torch.log1p(-torch.exp(chosen_lp))
    log_odds_rejected = rejected_lp - torch.log1p(-torch.exp(rejected_lp))
    
    log_ratio = log_odds_chosen - log_odds_rejected   # positive = model prefers chosen
    or_loss = -F.logsigmoid(log_ratio).mean()
    
    total_loss = sft_loss + lam * or_loss
    return total_loss, sft_loss.item(), or_loss.item()

# Verify with toy examples
# Case 1: model strongly prefers chosen
strong_chosen   = torch.tensor([-0.3, -0.2, -0.4])  # high probability
strong_rejected = torch.tensor([-3.0, -2.8, -3.2])  # low probability
loss1, sft1, or1 = orpo_loss(strong_chosen, strong_rejected)

# Case 2: model confused (similar probabilities)
confused_chosen   = torch.tensor([-1.5, -1.6, -1.4])
confused_rejected = torch.tensor([-1.6, -1.5, -1.7])
loss2, sft2, or2 = orpo_loss(confused_chosen, confused_rejected)

print("Strong preference (correct behaviour):")
print(f"  SFT={sft1:.3f}  OR={or1:.3f}  Total={loss1:.3f}")
print()
print("Confused model (needs training):")
print(f"  SFT={sft2:.3f}  OR={or2:.3f}  Total={loss2:.3f}")
print()
print("Higher OR loss = model doesn't differentiate chosen vs rejected → needs training")

## Training ORPO on Qwen3-0.6B

We use `Qwen/Qwen3-0.6B` — 596M parameters, ungated, fast.
The dataset format is identical to DPO: `{prompt, chosen, rejected}`.

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from torch.utils.data import DataLoader

gc.collect(); torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM"))
model.print_trainable_parameters()
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

In [ ]:
# Dataset: prompt + chosen (good response) + rejected (bad response)
raw_data = [
    {"prompt": "What is LoRA fine-tuning?",
     "chosen": "LoRA adds trainable low-rank matrices to frozen weights, reducing trainable parameters by 99% while maintaining quality.",
     "rejected": "LoRA is a Japanese animation series about a magical girl."},
    {"prompt": "Explain the attention mechanism.",
     "chosen": "Attention computes Attention(Q,K,V)=softmax(QK^T/√d_k)·V. Each token attends to all others, enabling long-range dependency capture.",
     "rejected": "Attention means focusing carefully on what someone says to you."},
    {"prompt": "What is gradient descent?",
     "chosen": "Gradient descent updates parameters θ ← θ - α·∇L(θ), iteratively minimizing a loss function by following the negative gradient.",
     "rejected": "Gradient descent is a hiking technique used when going downhill on steep terrain."},
    {"prompt": "What is a transformer architecture?",
     "chosen": "Transformers use stacked self-attention layers to process sequences in parallel. The architecture enables LLMs like GPT, BERT, and LLaMA.",
     "rejected": "A transformer is an electrical device that steps voltage up or down using electromagnetic induction."},
] * 8

class ORPODataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, max_len=192):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def encode(self, text):
        return tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=self.max_len, padding="max_length")

    def __getitem__(self, i):
        d = self.data[i]
        chosen_enc   = self.encode(d["prompt"] + " " + d["chosen"])
        rejected_enc = self.encode(d["prompt"] + " " + d["rejected"])
        return {
            "chosen_input_ids":       chosen_enc["input_ids"].squeeze(),
            "chosen_attention_mask":  chosen_enc["attention_mask"].squeeze(),
            "rejected_input_ids":     rejected_enc["input_ids"].squeeze(),
            "rejected_attention_mask": rejected_enc["attention_mask"].squeeze(),
        }

dataset = ORPODataset(raw_data, tokenizer)
print(f"Dataset: {len(dataset)} preference pairs")

# Training loop
opt = torch.optim.AdamW(model.parameters(), lr=8e-6)
model.train()

history = []
for step, batch in enumerate(DataLoader(dataset, batch_size=2, shuffle=True)):
    if step >= 16: break

    chosen_ids   = batch["chosen_input_ids"].cuda()
    chosen_mask  = batch["chosen_attention_mask"].cuda()
    rejected_ids = batch["rejected_input_ids"].cuda()
    rejected_mask = batch["rejected_attention_mask"].cuda()

    with torch.amp.autocast("cuda"):
        chosen_out   = model(input_ids=chosen_ids,   attention_mask=chosen_mask,   labels=chosen_ids)
        rejected_out = model(input_ids=rejected_ids, attention_mask=rejected_mask, labels=rejected_ids)
        # Convert per-batch loss to mean log-prob
        chosen_lp   = -chosen_out.loss.unsqueeze(0)
        rejected_lp = -rejected_out.loss.unsqueeze(0)
        loss, sft, or_ = orpo_loss(chosen_lp, rejected_lp, lam=0.1)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); opt.zero_grad()
    history.append({"loss": loss.item(), "sft": sft, "or": or_})
    if step % 4 == 3:
        print(f"Step {step+1:2d}: total={loss.item():.4f}  sft={sft:.4f}  or={or_:.4f}")

print(f"\n✅ ORPO training complete!")
print(f"   Loss: {history[0]['loss']:.4f} → {history[-1]['loss']:.4f}")
print(f"   Key advantage: NO reference model needed → ~50% less memory than DPO")

## Key Takeaways

1. **ORPO = SFT + odds ratio in one loss** — no reference model, no separate passes
2. **~50% less memory than DPO** — only one model in memory at all times
3. **Same dataset format as DPO** — `{prompt, chosen, rejected}` pairs
4. **λ=0.1 controls OR strength** — too high and OR dominates SFT, too low and preference signal is weak
5. **TRL API**: in TRL 1.3+, ORPO is in `trl.trainer` — check your version before importing

## When to Use ORPO vs DPO

| | ORPO | DPO |
|---|---|---|
| Reference model | Not needed | Required (frozen copy) |
| Memory | ~1x model | ~2x model |
| Stability | Slightly less stable | More stable |
| Quality | Comparable to DPO | Baseline |
| Best for | Memory-constrained fine-tuning | Quality-critical alignment |